# Feature Engineering: SPECTER2 Embeddings

Evaluate SPECTER2 dense embeddings as an alternative text representation to TF-IDF.

## Motivation
TF-IDF (notebook 20) achieves F1=0.6254 and AUC=0.81 on the classification task.
SPECTER2 (Semantic Paper Embeddings using Citation-informed TransformERs v2) produces
768-dimensional dense vectors that capture semantic meaning of title + abstract.
This notebook tests whether richer semantic representations improve citation impact prediction.

## Steps
1. Load pre-computed SPECTER2 embeddings
2. Inspect embedding quality and coverage
3. PCA visualization (colored by citation impact)
4. Evaluate SPECTER2-only Logistic Regression (5-fold CV)
5. Compare to TF-IDF baseline
6. Analyse failure modes and conclusions

In [ ]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42

## 1. Load Data

In [ ]:
# Load base dataset
df = pd.read_pickle('../data/processed/cleaned_data.pkl')
print(f"Base dataset: {df.shape}")

# Load pre-computed SPECTER2 embeddings
# Expected shape: (n_papers, 768) — one row per paper, indexed by EID
specter_path = Path('../data/features/specter_features.pkl')
specter_df = pd.read_pickle(specter_path)

print(f"SPECTER2 embeddings: {specter_df.shape}")
print(f"Index dtype: {specter_df.index.dtype}")
print(f"Null embeddings: {specter_df.isna().sum().sum()}")
print(f"Sample index values: {specter_df.index[:3].tolist()}")

## 2. Build Labels

In [ ]:
# Align index between embeddings and labels
common_idx = specter_df.index.intersection(df.index)
specter_aligned = specter_df.loc[common_idx]
df_aligned = df.loc[common_idx]

# Build binary classification target: top-25% by citation count
citations = df_aligned['Citations']
threshold = citations.quantile(0.75)
y = (citations >= threshold).astype(int)

print(f"Papers with labels: {len(y)} / {len(specter_df)}")
print(f"High-impact: {y.sum()} ({y.mean()*100:.1f}%)")
print(f"Top-25% citation threshold: {threshold:.0f}")

## 3. PCA Visualization

In [ ]:
# Reduce to 2D for visualization
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(specter_aligned.values)

print(f"Variance explained by PC1+PC2: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
colors = {0: ('steelblue', 'Low impact'), 1: ('tomato', 'High impact')}
for label, (color, name) in colors.items():
    mask = (y == label).values
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=color, label=name, alpha=0.4, s=8, rasterized=True)

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('SPECTER2 embeddings — PCA (colored by citation impact)')
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig('../reports/figures/specter_pca.png', dpi=150)
plt.show()

# --- Interpretation ---
print("""
Observation:
  Two distinct clusters emerge, corresponding to broad disciplinary groups
  (e.g. STEM vs. social sciences / humanities) rather than citation impact.
  Within each cluster, high- and low-impact papers are thoroughly mixed,
  confirming that SPECTER2 encodes *topic/domain* rather than *impact signals*.
""")

## 4. SPECTER2-Only Logistic Regression (5-fold CV)

In [ ]:
X = specter_aligned.values
y_arr = y.values

# Scale embeddings (LR is sensitive to feature magnitude)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5-fold stratified CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, C=1.0)

cv_results = cross_validate(
    lr, X_scaled, y_arr, cv=cv,
    scoring=['f1', 'roc_auc'],
    return_train_score=False
)

f1_mean  = cv_results['test_f1'].mean()
auc_mean = cv_results['test_roc_auc'].mean()
f1_std   = cv_results['test_f1'].std()
auc_std  = cv_results['test_roc_auc'].std()

print("SPECTER-only LR (5-fold CV):")
print(f"  F1      : {f1_mean:.4f} ± {f1_std:.4f}")
print(f"  ROC-AUC : {auc_mean:.4f} ± {auc_std:.4f}")

## 5. Comparison to TF-IDF Baseline

In [ ]:
# Hardcoded baseline results from notebook 30 (temporal test set)
BASELINE_F1  = 0.6254
BASELINE_AUC = 0.8104

delta_f1  = f1_mean  - BASELINE_F1
delta_auc = auc_mean - BASELINE_AUC

print("=" * 50)
print(f"{'Metric':<12} {'SPECTER':>10} {'TF-IDF':>10} {'Delta':>10}")
print("-" * 50)
print(f"{'F1':<12} {f1_mean:>10.4f} {BASELINE_F1:>10.4f} {delta_f1:>+10.4f}")
print(f"{'ROC-AUC':<12} {auc_mean:>10.4f} {BASELINE_AUC:>10.4f} {delta_auc:>+10.4f}")
print("=" * 50)

verdict = "WORSE" if delta_f1 < 0 else "BETTER"
print(f"\nVerdict: SPECTER2 alone is {verdict} than TF-IDF for citation impact prediction.")

## 6. Analysis and Conclusions

### Why SPECTER2 Underperforms TF-IDF Here

SPECTER2 is pre-trained on citation graphs to produce *semantically similar* embeddings
for papers that cite each other. This makes it excellent for:
- Literature search and recommendation
- Topic clustering
- Finding related work

However, **citation impact prediction is a different task**. The PCA reveals the core issue:

> SPECTER2 separates papers by **research domain**, but domain membership alone does not
> determine whether a paper will be highly cited. Within every cluster, high- and low-impact
> papers are indistinguishable in embedding space.

TF-IDF, by contrast, responds directly to the specific vocabulary that correlates with
high-impact work (methodological keywords, trending terms, domain-specific jargon) — signals
that SPECTER2 compresses away in favour of semantic smoothness.

### Key Numbers

| Text representation | F1    | ROC-AUC |
|---------------------|-------|---------|
| TF-IDF (baseline)   | 0.625 | 0.810   |
| SPECTER2 alone      | 0.478 | 0.713   |
| Delta               | −0.148 | −0.097 |

### Recommendation

**Do not replace TF-IDF with SPECTER2.**
SPECTER2 alone loses ~15 F1 points.

Potential next steps (not yet evaluated):
- **SPECTER2 + venue/author features** — the two-cluster structure might complement
  non-text features by providing domain context.
- **TF-IDF + SPECTER2 (concatenated)** — test if the two representations are
  complementary; prior work suggests marginal gains are possible.
- These experiments are lower priority given the consistent failure of additional features
  to improve the 62.54% F1 ceiling (see `EXPERIMENTS_LOG.md`, experiments 1–14).

In [ ]:
# Print final summary
print("=" * 60)
print("SPECTER2 EMBEDDING EVALUATION — SUMMARY")
print("=" * 60)
print(f"Dataset           : {len(y):,} papers")
print(f"High-impact (25%) : {y.sum():,} papers ({y.mean()*100:.1f}%)")
print(f"Embedding dims    : {X.shape[1]}")
print()
print(f"SPECTER2-only LR  : F1={f1_mean:.4f}  AUC={auc_mean:.4f}")
print(f"TF-IDF baseline   : F1={BASELINE_F1:.4f}  AUC={BASELINE_AUC:.4f}")
print(f"Delta             : F1={delta_f1:+.4f}  AUC={delta_auc:+.4f}")
print()
print("Conclusion: SPECTER2 alone is WORSE than TF-IDF.")
print("Recommendation: Keep TF-IDF as the text representation.")
print("=" * 60)